# Father and Daughter — train LoRA tren Colab T4

Chay **mot lan** cho moi nhan vat. Khoang **60-90 phut/LoRA** tren T4.
Ba LoRA (`dad`, `girl`, `style`) chia lam 2 phien Colab.

**Truoc khi mo notebook nay**, tren may 1 chay:
```
python scripts/prep_dataset.py --kind girl --src raw/moonie --out training/girl
rclone sync training gdrive:father-and-daughter/training
```
va xu ly het canh bao script in ra. **Thieu anh thi LoRA khong giu duoc
nhan vat** — do la that bai pho bien nhat, khong phai do hyperparameter.

Moonie 2 tuoi can **>= 35 anh**: mat tre nho it dac diem rieng nen model
rat de truot tuoi len 4-5.

In [ ]:
# 1. GPU. Phai co, va phai biet no khong ho tro bf16.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('bf16:', torch.cuda.is_bf16_supported(), '-> T4 la False, nen bat buoc dung fp16')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BRIDGE = '/content/drive/MyDrive/father-and-daughter'
!ls {BRIDGE}/training/

In [ ]:
# 2. Cai dat (~5 phut). diffusers ban git vi script train nam trong repo.
!pip install -q git+https://github.com/huggingface/diffusers
!pip install -q accelerate transformers peft bitsandbytes safetensors xformers datasets
!wget -q -O /content/train_lora_sdxl.py https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py
import os; os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'

## 3. Chon LoRA can train
Doi `KIND` roi chay cac cell duoi. Xong mot cai thi quay lai doi `KIND`
va chay lai — **Runtime > Restart** truoc moi lan de giai phong VRAM.

In [ ]:
KIND = 'girl'          # 'dad' | 'girl' | 'mother' | 'style'

CFG = {
    'dad':    ('f4thr_man, a man',           1400),
    'girl':   ('m00nie_girl, a toddler girl', 1800),  # 2 tuoi can nhieu step hon
    'mother': ('m0ther_woman, a woman',      1400),
    'style':  ('cozyhn_style, an interior scene', 1600),
}
PROMPT, STEPS = CFG[KIND]
DATA = f'{BRIDGE}/training/{KIND}'
OUT  = f'/content/out_{KIND}'

import glob
n = len(glob.glob(DATA + '/*.png'))
print(f'{KIND}: {n} anh | prompt: {PROMPT!r} | {STEPS} step')
assert n >= 20, f'Chi co {n} anh. Quay lai chay prep_dataset.py, dung train voi it anh hon.'

In [ ]:
# 4. TRAIN. ~60-90 phut tren T4. Dung dong tab.
#
# Vi sao tung tham so:
#   resolution 768  - 1024 khong du VRAM tren T4 16GB
#   fp16            - T4 la Turing, khong co bf16
#   sdxl-vae-fp16-fix - VAE goc cua SDXL sinh NaN khi chay fp16
#   8bit adam + gradient checkpointing - deu bat buoc de vua VRAM
#   khong train text encoder - tiet kiem rat nhieu VRAM, voi 1 nhan vat
#                              thi trigger token rieng da du
!accelerate launch /content/train_lora_sdxl.py \
  --pretrained_model_name_or_path='stabilityai/stable-diffusion-xl-base-1.0' \
  --pretrained_vae_model_name_or_path='madebyollin/sdxl-vae-fp16-fix' \
  --instance_data_dir='{DATA}' \
  --instance_prompt='{PROMPT}' \
  --output_dir='{OUT}' \
  --resolution=768 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --use_8bit_adam \
  --enable_xformers_memory_efficient_attention \
  --cache_latents \
  --mixed_precision='fp16' \
  --learning_rate=1e-4 \
  --lr_scheduler='cosine' \
  --lr_warmup_steps=100 \
  --rank=32 \
  --max_train_steps={STEPS} \
  --checkpointing_steps=400 \
  --seed=42

In [ ]:
# 5. Luu ngay ra Drive truoc khi lam gi khac — Colab co the ngat bat cu luc nao.
import shutil, os, glob
os.makedirs(f'{BRIDGE}/loras', exist_ok=True)
src = glob.glob(f'{OUT}/pytorch_lora_weights.safetensors')[0]
dst = f'{BRIDGE}/loras/lora_{KIND}.safetensors'
shutil.copy2(src, dst)
print(dst, os.path.getsize(dst)//1024//1024, 'MB')

## 6. Kiem tra LoRA — dung bo qua buoc nay
Nhin ky 4 anh sinh ra:
- Nhan vat co **giong nhau giua 4 anh** khong? Khong giong = thieu anh train.
- Moonie co **dung 2 tuoi** khong, hay bi gia di thanh 4-5?
- Co bi **dinh cung mot tu the/bo cuc** khong? Dinh = overfit, giam step.

In [ ]:
import torch, gc
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
gc.collect(); torch.cuda.empty_cache()

vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', vae=vae,
    torch_dtype=torch.float16, variant='fp16', use_safetensors=True)
pipe.enable_model_cpu_offload()
pipe.load_lora_weights(dst)

TESTS = {
 'girl':  'm00nie_girl sitting on a small red plastic stool on a balcony, holding a cream rabbit plush, warm afternoon light',
 'dad':   'f4thr_man standing in a small kitchen holding a chipped white ceramic mug, morning light through aluminium window',
 'mother':'m0ther_woman folding laundry near a balcony drying rack, soft overcast light',
 'style': 'cozyhn_style, empty living room with bamboo mat and an old standing fan, late afternoon sun',
}
imgs = pipe([TESTS[KIND]]*4, negative_prompt=['photo, 3d render, text, deformed hands']*4,
            width=768, height=1344, num_inference_steps=30, guidance_scale=6.0,
            generator=[torch.Generator('cuda').manual_seed(s) for s in (1,2,3,4)]).images

from PIL import Image
g = Image.new('RGB', (768*2//2, 1344*2//2))
for i, im in enumerate(imgs):
    g.paste(im.resize((384,672)), ((i%2)*384, (i//2)*672))
g.save(f'{BRIDGE}/loras/preview_{KIND}.jpg', quality=90)
g

## Xong mot LoRA
**Runtime > Restart runtime**, doi `KIND`, chay lai tu cell 1.

Train het 3 LoRA roi thi quay ve notebook `Father_and_Daughter_Worker.ipynb`
— no tu nap `lora_dad`, `lora_girl`, `lora_style` tu `{BRIDGE}/loras/`.

**Neu nhan vat khong nhat quan:** them anh train truoc, dung tang step.
Tang step tren dataset ngheo chi lam overfit nang hon.